In [42]:
import torch
import pandas as pd
import os
from rinalmo.pretrained import get_pretrained_model
DEVICE = "cuda:0"
import numpy as np
np.random.seed(42)

In [2]:
model, alphabet = get_pretrained_model(model_name="giga-v1")
model = model.to(device=DEVICE)
model.eval()

RiNALMo(
  (embedding): Embedding(22, 1280, padding_idx=1)
  (transformer): Transformer(
    (blocks): ModuleList(
      (0-32): 33 x TransformerBlock(
        (mh_attn): FlashMultiHeadSelfAttention(
          (rotary_emb): RotaryEmbedding()
          (flash_self_attn): FlashAttention()
          (Wqkv): Linear(in_features=1280, out_features=3840, bias=False)
          (attention_dropout): Dropout(p=0.1, inplace=False)
          (out_proj): Linear(in_features=1280, out_features=1280, bias=False)
        )
        (attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
        (transition): Sequential(
          (0): SwiGLU(
            (linear): Linear(in_features=1280, out_features=3413, bias=True)
            (linear_gate): Linear(in_features=1280, out_features=3413, bias=True)
          )
          (1): Dropout(p=0.0, inplace=False)
          (2): Linear(in_features=3413, out_features=1280, bias=True)
        )
        (out_layer_norm): LayerNorm((1280,), eps=1e-05

In [3]:
rfam_ids = pd.read_csv("sequence_rfam_mapping_annotations.csv") #their corresponding rfam ids (family)
alignment_score_matrix = pd.read_csv("alignment_matrix_3_perfam.csv", index_col=0) #pairwise alignment matrix aren shared with me
ids = alignment_score_matrix.index.tolist() #get indices which are in row and column names 
fasta_df = pd.read_csv("fasta_df.csv") #contains sequence_id, sequence, and length for every sequence
fasta_df = fasta_df.iloc[ids] #only keep the ones from ids
rfam_ids = rfam_ids.iloc[ids] #only keep the ones from ids

In [4]:
eukaryotic_indexes = rfam_ids.index[rfam_ids["rfam_id"] == "RF01960"]
eukarytoic_values = fasta_df.loc[eukaryotic_indexes, "sequence"].tolist()

In [15]:
seqs = eukarytoic_values[2]

In [17]:
tokens = torch.tensor(alphabet.batch_tokenize([seqs]), dtype=torch.int64, device=DEVICE) # FIXED
slider = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
nucleotides = {i: [] for i in range(1, len(seqs)+1)}

In [18]:
while True:
    slider_checked = [x for x in slider if x in nucleotides]
    if slider_checked:
        masked_tokens = tokens.clone()
        masked_tokens[0, slider_checked] = alphabet.mask_idx
        with torch.no_grad(), torch.cuda.amp.autocast():
            outputs = model(masked_tokens)
        logits = outputs["logits"]
        mask_logits = logits[0, slider_checked]
        predicted_token_ids = torch.argmax(mask_logits, dim=-1)
        predicted_nucleotides = [alphabet.idx_to_tkn[i.item()] for i in predicted_token_ids]
        for pos, nuc in zip(slider_checked, predicted_nucleotides):
            nucleotides[pos].append(nuc)
        slider = [x + 1 for x in slider]
    else:  
        break    

In [28]:
consensus_chars = []
for pos in nucleotides.keys():
    guesses = nucleotides[pos]
    if guesses:
        most_common = max(set(guesses), key=guesses.count)
        consensus_chars.append(most_common)
consensus_sequence = "".join(consensus_chars)

variable_keys = []
for pos, guesses in nucleotides.items():
    if len(set(guesses)) > 1:
        variable_keys.append(pos)
print(f"Consensus Sequence: {consensus_sequence}")
print(f"Variable Positions: {sorted(variable_keys)}")

Consensus Sequence: TTGGTTGATCTTGCCAGTAGCATATGCTTGTCTCAAAGATTAAGCCATGCATGTCTAAGTACACACGGTTTGAAAAGTGAAACTGCGAATGTTAAATTAAATTAGGTTCCTTTGATCCGACAATGTTACTTGGATAACTGTGGCAATTCTAGAGCTAATACATGCAAACAAGCGCTGACCTCCGGGGATGCGTGCATTTATTAGACCAAAAACCAGCGGGGCGGTCCGGGGGCCCGCTGCTTTGGTGACTCTTGATAACCTTGGGCGGATCGCACGGCCTTTGTGGCGGCGACGTCTCATTCGAATGTCTGCCCTATCAACTTTCGATGGTACTTTCTGTGCCTACCATGGTGACCACGGGTAACGGGGAATCAGGGTTCGATTCCGGAGAGGGAGCCTGAGAAATGGCTACCACATCCAAGGAAGGAAGCAGAAGGGCCCCAAATTACCCACTCCCGACACGGGGGAGGTGTGAAAAAATATAAAAACAGGACTCTTTGAGGCCTGTAATTGGAATGAGTACACTTTAAATCCTTTAACGATGATCAATTGGAGGGCAAGTTTGTAAAAAAATT
Variable Positions: [11, 12, 13, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 27, 28, 64, 65, 67, 68, 69, 70, 72, 73, 75, 76, 77, 78, 79, 80, 81, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 98, 100, 101, 103, 104, 105, 106, 108, 109, 110, 111, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 126, 151, 162, 167, 169, 170, 176, 182, 183, 184, 189, 191, 192, 196, 200, 201, 202, 203, 205

In [20]:
mismatches = []
for i, (orig, cons) in enumerate(zip(seqs, consensus_sequence)):
    if orig != cons:
        mismatches.append({
            'position': i + 1, # +1 for 1-based biological indexing
            'original': orig,
            'consensus': cons
        })

In [35]:
mismatches

[{'position': 1, 'original': 'C', 'consensus': 'T'},
 {'position': 11, 'original': 'C', 'consensus': 'T'},
 {'position': 13, 'original': 'C', 'consensus': 'G'},
 {'position': 15, 'original': 'T', 'consensus': 'C'},
 {'position': 21, 'original': 'T', 'consensus': 'C'},
 {'position': 31, 'original': 'A', 'consensus': 'T'},
 {'position': 43, 'original': 'G', 'consensus': 'A'},
 {'position': 52, 'original': 'A', 'consensus': 'T'},
 {'position': 64, 'original': 'A', 'consensus': 'C'},
 {'position': 65, 'original': 'T', 'consensus': 'A'},
 {'position': 66, 'original': 'T', 'consensus': 'C'},
 {'position': 68, 'original': 'A', 'consensus': 'G'},
 {'position': 69, 'original': 'C', 'consensus': 'T'},
 {'position': 71, 'original': 'A', 'consensus': 'T'},
 {'position': 73, 'original': 'T', 'consensus': 'A'},
 {'position': 75, 'original': 'C', 'consensus': 'A'},
 {'position': 86, 'original': 'T', 'consensus': 'C'},
 {'position': 92, 'original': 'G', 'consensus': 'T'},
 {'position': 93, 'original':

In [64]:
#I will make some sequences up same length as the above 3
for i in eukarytoic_values: print(len(i))
sequence_first = "".join(np.random.choice(list("AUCG"), size=763))
sequence_second = "".join(np.random.choice(list("AUCG"), size=781))
sequence_third = "".join(np.random.choice(list("AUCG"), size=575))

763
781
575


In [75]:
seqs = sequence_third
seqs

'CUAUGCAGAAGUGGUCGAGGAUCCAUGGGGGAGGUGCAAACGACUGCGUCGCAGCAUGGUAUGCAAAUUCCACUCUCGUGUCGCUGACUGCGCCCGGCAAAUAGAGUGGGACGAAACCCCUCCCGGCUGAGAGCGCCGAAGCAUCGCUUAUCAUAAGAAUCCUCAGAAGUAAUAACUAAGGGAAGCCCCUGAGCAUGAACCGUACGCGCUACAGUACAAGUUAAGAGUUGAUCGUUUCCCAGACGAAACUGCGCAAUUGUUGCUCAUAACGUCGAUAAUGGUUCCAUGAAAAGAUCGUUUCCCUCACGUUACCGAGGUUCCGCGCCUUUUAUAUCGCCAUGGUAAAGAAGGCGUGGACACAUAAUUGAUUGCUUAGACGAUUGUAGUUCCGCAUUUGGCUUUUUAUAUGGAUGGUUGUCUGGACGCUGAGACAUUUGUUGCUUGCCCGAAUUUAGUGUGGAGCAUACCGCCCCUUAGUCGUACGUGUACACCGUAAAGGGCACAAUAGGGUUCAAGGGGGGAUGAGCCAUCCUCGUCUGCACCGAAAGGGGGAUGUCCGCUCCUCGUUAAUUAGC'

In [76]:
tokens = torch.tensor(alphabet.batch_tokenize([seqs]), dtype=torch.int64, device=DEVICE) # FIXED
slider = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
nucleotides = {i: [] for i in range(1, len(seqs)+1)}
while True:
    slider_checked = [x for x in slider if x in nucleotides]
    if slider_checked:
        masked_tokens = tokens.clone()
        masked_tokens[0, slider_checked] = alphabet.mask_idx
        with torch.no_grad(), torch.cuda.amp.autocast():
            outputs = model(masked_tokens)
        logits = outputs["logits"]
        mask_logits = logits[0, slider_checked]
        predicted_token_ids = torch.argmax(mask_logits, dim=-1)
        predicted_nucleotides = [alphabet.idx_to_tkn[i.item()] for i in predicted_token_ids]
        for pos, nuc in zip(slider_checked, predicted_nucleotides):
            nucleotides[pos].append(nuc)
        slider = [x + 1 for x in slider]
    else:  
        break    

In [77]:
consensus_chars = []
for pos in nucleotides.keys():
    guesses = nucleotides[pos]
    if guesses:
        most_common = max(set(guesses), key=guesses.count)
        consensus_chars.append(most_common)
consensus_sequence = "".join(consensus_chars)

variable_keys = []
for pos, guesses in nucleotides.items():
    if len(set(guesses)) > 1:
        variable_keys.append(pos)
print(f"Consensus Sequence: {consensus_sequence}")
print(f"Variable Positions: {sorted(variable_keys)}")

Consensus Sequence: CCGCAGGTAAAACGAAGGGGACGTCGGCCACCACCGCCGCCGCGGCAACGGCGAACGCGGCGGCGGCGGCGGCGGCGACGACGAAAAAAAAAACGGCGACGGCGGCGGCGAGAAAGAAGAAGAACAAAAAAAAGAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAACAACAACAGCAACAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAGAACAAAAAAAAAAAAAAAAAAAAAAAATTAACGATGCATAAATTCAGTAATAAAAAAAGAAAAAAATAAAAATTAAAAAAAAACAAAAACAAAAACAACAAAATTAAAAAAATTTTAAAAAAAAATATTATTATATTTATTATTATAAAAAAAAAAATTTTATTATTTATTTTTTTATTTTTGTATTTTTTTTTATTTTAAAAATTTTTTTTTTGATAAATAATAATATGTTGAATCAGCCGCATTAATTTCGGCCAACGGGGCAAAAAAAAAGAAAAACGAGGGGGACACCAGAACAAAAAACAACAAGATCGAAAAAAAAAACGCCGCCACCAAAAAAAAAACCGCAAACTCCAC
Variable Positions: [3, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 70, 71, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 103,

In [78]:
mismatches = []
for i, (orig, cons) in enumerate(zip(seqs, consensus_sequence)):
    if orig != cons:
        mismatches.append({
            'position': i + 1, # +1 for 1-based biological indexing
            'original': orig,
            'consensus': cons
        })
mismatches

[{'position': 2, 'original': 'U', 'consensus': 'C'},
 {'position': 3, 'original': 'A', 'consensus': 'G'},
 {'position': 4, 'original': 'U', 'consensus': 'C'},
 {'position': 5, 'original': 'G', 'consensus': 'A'},
 {'position': 6, 'original': 'C', 'consensus': 'G'},
 {'position': 7, 'original': 'A', 'consensus': 'G'},
 {'position': 8, 'original': 'G', 'consensus': 'T'},
 {'position': 11, 'original': 'G', 'consensus': 'A'},
 {'position': 12, 'original': 'U', 'consensus': 'A'},
 {'position': 13, 'original': 'G', 'consensus': 'C'},
 {'position': 15, 'original': 'U', 'consensus': 'A'},
 {'position': 16, 'original': 'C', 'consensus': 'A'},
 {'position': 18, 'original': 'A', 'consensus': 'G'},
 {'position': 22, 'original': 'U', 'consensus': 'C'},
 {'position': 23, 'original': 'C', 'consensus': 'G'},
 {'position': 24, 'original': 'C', 'consensus': 'T'},
 {'position': 25, 'original': 'A', 'consensus': 'C'},
 {'position': 26, 'original': 'U', 'consensus': 'G'},
 {'position': 28, 'original': 'G', 